# 0.10 - Unified Grade 7 Student Flows (Diagnostic Version)

**Purpose:** Build unified Grade 7 flows dataset AND diagnose low coordinate availability (5.1%)

**Critical Issue:** Only 5.1% of records have complete coordinate data. This notebook systematically investigates:
1. Coordinate source file coverage
2. School ID matching issues (dtype, format, alignment)
3. Merge logic in modules
4. Node table integration
5. Coordinate flow through the pipeline

**Author:** Claude Code  
**Date:** 2025-11-17

# 0. Setup

In [11]:
import os, sys
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import json
from importlib import reload

# Add project root to Python path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import config and setup
from config import setup_notebook, get_path
setup_notebook()

print("✓ Setup complete")

✓ Project root: /workspace/project_paaral
✓ Working directory: /workspace/project_paaral
✓ Python path updated
✓ Setup complete


In [18]:
# Import modules
from modules import gr7_enrollees_processor, unified_gr7_flow_builder
from modules import public_coordinates_processor, private_coordinates_processor

reload(gr7_enrollees_processor)
reload(unified_gr7_flow_builder)
reload(public_coordinates_processor)
reload(private_coordinates_processor)

from modules.gr7_enrollees_processor import Gr7EnrolleesProcessor
from modules.unified_gr7_flow_builder import UnifiedGr7FlowBuilder
from modules.public_coordinates_processor import PublicSchoolsProcessor
from modules.private_coordinates_processor import PrivateSchoolsProcessor

# 1. DIAGNOSTIC: Coordinate Source File Analysis

**Goal:** Understand what schools have coordinates in the source files

In [21]:
print("="*80)
print("STEP 1: PUBLIC COORDINATE SOURCE FILE")
print("="*80)

# Load coordinate source directly
coord_processor = PublicSchoolsProcessor(verbose=True)
coord_raw = coord_processor.load_data()
coord_processed = coord_processor.process()

STEP 1: PUBLIC COORDINATE SOURCE FILE


INFO:modules.public_coordinates_processor:Available sheets: ['DB']
INFO:modules.public_coordinates_processor:Using sheet: DB
INFO:modules.public_coordinates_processor:Reading Excel file with header at row 6 (0-indexed row 5), skipping first 5 rows
INFO:modules.public_coordinates_processor:Detected columns from header row: ['Region', 'Division', 'District', 'LIS SCHOOL ID', 'NSBI SCHOOL ID', 'School Name', 'Street Address', 'Province', 'Municipality', 'Legislative District', 'Barangay', 'Longitude', 'Latitude']
INFO:modules.public_coordinates_processor:Removed 0 completely empty rows
INFO:modules.public_coordinates_processor:Found coordinate columns: ["Latitude-like: 'Legislative District'", "Longitude-like: 'Longitude'", "Latitude-like: 'Latitude'"]
INFO:modules.public_coordinates_processor:Successfully loaded 47821 records with 13 columns
INFO:modules.public_coordinates_processor:All columns: ['Region', 'Division', 'District', 'LIS SCHOOL ID', 'NSBI SCHOOL ID', 'School Name', 'Street 

In [22]:
display(coord_processed.head(1))

,region,division,district,lis_school_id,nsbi_school_id,school_name,street_address,province,municipality,legislative_district,barangay,longitude,latitude,coord_valid,coord_missing,coord_out_of_bounds,coord_potentially_switched,school_id_processed
0,Region I,Ilocos Norte,Bacarra I,100001,100001.0,Apaleng-Libtong ES,"Brgy. 21, Libtong, Bacarra, Ilocos Norte",ILOCOS NORTE,BACARRA,1st District,LIBTONG,120.614372,18.26686,True,False,False,False,100001


In [23]:
print(f"\nTotal schools in coordinate file: {len(coord_processed):,}")
print(f"Schools with valid coordinates: {(coord_processed['coord_valid'] == True).sum():,}")
print(f"Valid coordinate rate: {(coord_processed['coord_valid'] == True).sum()/len(coord_processed)*100:.1f}%")

# Inspect school_id column
print(f"\nSchool ID column dtype: {coord_processed['school_id_processed'].dtype}")
print(f"Sample school IDs: {coord_processed['school_id_processed'].head(10).tolist()}")
print(f"Unique school IDs: {coord_processed['school_id_processed'].nunique():,}")


Total schools in coordinate file: 47,821
Schools with valid coordinates: 47,116
Valid coordinate rate: 98.5%

School ID column dtype: object
Sample school IDs: ['100001', '100002', '100003', '100004', '100005', '100006', '100007', '100008', '100009', '100010']
Unique school IDs: 47,821


In [24]:
# Standardize school_id to string for matching
coord_processed['school_id_processed'] = coord_processed['school_id_processed'].astype(str).str.strip()

# Create sets of school IDs for later comparison
all_coord_school_ids = set(coord_processed['school_id_processed'].unique())
valid_coord_school_ids = set(coord_processed[coord_processed['coord_valid'] == True]['school_id_processed'].unique())

print(f"\nSchool ID sets created:")
print(f"  All schools in coordinate file: {len(all_coord_school_ids):,}")
print(f"  Schools with valid coordinates: {len(valid_coord_school_ids):,}")

# Inspect a few schools with coordinates
print(f"\nSample schools WITH valid coordinates:")
sample_valid = coord_processed[coord_processed['coord_valid'] == True][['school_id_processed', 'school_name', 'latitude', 'longitude']].head(3)
display(sample_valid)

print(f"\nSample schools WITHOUT valid coordinates:")
sample_invalid = coord_processed[coord_processed['coord_valid'] == False][['school_id_processed', 'school_name', 'latitude', 'longitude']].head(3)
display(sample_invalid)


School ID sets created:
  All schools in coordinate file: 47,821
  Schools with valid coordinates: 47,116

Sample schools WITH valid coordinates:


,school_id_processed,school_name,latitude,longitude
0,100001,Apaleng-Libtong ES,18.266860,120.614372
1,100002,Bacarra CES,18.251272,120.609487
2,100003,Buyon ES,18.234670,120.616050



Sample schools WITHOUT valid coordinates:


,school_id_processed,school_name,latitude,longitude
1311,101219,San Nicolas Elementary School,NaN,NaN
1390,101276,Maigpa ES,NaN,NaN
1529,101389,Dimantal E/S,NaN,NaN


# 2. DIAGNOSTIC: Grade 7 Enrollees File Analysis

**Goal:** Understand what schools are in the Grade 7 enrollees dataset

In [25]:
print("="*80)
print("STEP 2: GRADE 7 ENROLLEES RAW FILE")
print("="*80)

# Load raw enrollees file to inspect before processing
gr7_file_path = 'data/public/SY 2023-2024 Gr 7 Enrollees.xlsx'

# Read first few rows to inspect structure
gr7_raw = pd.read_excel(
    gr7_file_path,
    usecols=[
        'Region', 'Division',
        'School Namein Grade 7',  # Note: typo in source file
        'BEIS School ID in Grade 7',
        'School Name in Grade 6',
        'BEIS School ID in Grade 6'
    ],
    nrows=100000  # Load subset for quick inspection
)

STEP 2: GRADE 7 ENROLLEES RAW FILE


In [26]:
print(f"\nSample of raw Grade 7 enrollees (first 5 rows):")
display(gr7_raw.head())

print(f"\nSchool ID column dtypes:")
print(f"  School ID in Grade 6: {gr7_raw['BEIS School ID in Grade 6'].dtype}")
print(f"  School ID in Grade 7: {gr7_raw['BEIS School ID in Grade 7'].dtype}")

print(f"\nSample School IDs from Grade 6 column:")
print(gr7_raw['BEIS School ID in Grade 6'].head(10).tolist())

print(f"\nSample School IDs from Grade 7 column:")
print(gr7_raw['BEIS School ID in Grade 7'].head(10).tolist())


Sample of raw Grade 7 enrollees (first 5 rows):


,Region,Division,BEIS School ID in Grade 7,School Namein Grade 7,BEIS School ID in Grade 6,School Name in Grade 6
0,Region I,Ilocos Norte,300001,Adams National High School,100160,Adams CES
1,Region I,Ilocos Norte,300001,Adams National High School,100160,Adams CES
2,Region I,Ilocos Norte,300001,Adams National High School,100160,Adams CES
3,Region I,Ilocos Norte,300001,Adams National High School,100160,Adams CES
4,Region I,Ilocos Norte,300001,Adams National High School,100160,Adams CES



School ID column dtypes:
  School ID in Grade 6: int64
  School ID in Grade 7: int64

Sample School IDs from Grade 6 column:
[100160, 100160, 100160, 100160, 100160, 100160, 100160, 100160, 100160, 100160]

Sample School IDs from Grade 7 column:
[300001, 300001, 300001, 300001, 300001, 300001, 300001, 300001, 300001, 300001]


In [27]:
# Convert to string and check unique counts
gr7_raw['school_id_origin'] = gr7_raw['BEIS School ID in Grade 6'].astype(str).str.strip()
gr7_raw['school_id_destination'] = gr7_raw['BEIS School ID in Grade 7'].astype(str).str.strip()

origin_ids_raw = set(gr7_raw['school_id_origin'].dropna().unique())
dest_ids_raw = set(gr7_raw['school_id_destination'].dropna().unique())

print(f"\nUnique school IDs in raw Grade 7 enrollees (first 100K rows):")
print(f"  Origin (Grade 6) schools: {len(origin_ids_raw):,}")
print(f"  Destination (Grade 7) schools: {len(dest_ids_raw):,}")
print(f"  Total unique: {len(origin_ids_raw | dest_ids_raw):,}")


Unique school IDs in raw Grade 7 enrollees (first 100K rows):
  Origin (Grade 6) schools: 5,135
  Destination (Grade 7) schools: 1,098
  Total unique: 5,851


# 3. DIAGNOSTIC: School ID Overlap Analysis

**Goal:** Check if Grade 7 enrollee schools are present in coordinate file

In [28]:
print("="*80)
print("STEP 3: SCHOOL ID OVERLAP (RAW FILES)")
print("="*80)

# Check overlap
origin_in_coords = origin_ids_raw & all_coord_school_ids
dest_in_coords = dest_ids_raw & all_coord_school_ids

origin_in_valid_coords = origin_ids_raw & valid_coord_school_ids
dest_in_valid_coords = dest_ids_raw & valid_coord_school_ids

print(f"\n1. ORIGIN SCHOOLS (Grade 6):")
print(f"   Total origin school IDs: {len(origin_ids_raw):,}")
print(f"   Found in coordinate file: {len(origin_in_coords):,} ({len(origin_in_coords)/len(origin_ids_raw)*100:.1f}%)")
print(f"   With VALID coordinates: {len(origin_in_valid_coords):,} ({len(origin_in_valid_coords)/len(origin_ids_raw)*100:.1f}%)")

print(f"\n2. DESTINATION SCHOOLS (Grade 7):")
print(f"   Total destination school IDs: {len(dest_ids_raw):,}")
print(f"   Found in coordinate file: {len(dest_in_coords):,} ({len(dest_in_coords)/len(dest_ids_raw)*100:.1f}%)")
print(f"   With VALID coordinates: {len(dest_in_valid_coords):,} ({len(dest_in_valid_coords)/len(dest_ids_raw)*100:.1f}%)")

print(f"\n3. POTENTIAL RECORDS WITH BOTH COORDINATES:")
potential_complete = len(origin_in_valid_coords) * len(dest_in_valid_coords) / 10  # Rough estimate
print(f"   If matching was perfect, we could have ~{potential_complete:,.0f} records with coordinates")
print(f"   (This is a very rough upper bound estimate)")

STEP 3: SCHOOL ID OVERLAP (RAW FILES)

1. ORIGIN SCHOOLS (Grade 6):
   Total origin school IDs: 5,135
   Found in coordinate file: 4,459 (86.8%)
   With VALID coordinates: 4,443 (86.5%)

2. DESTINATION SCHOOLS (Grade 7):
   Total destination school IDs: 1,098
   Found in coordinate file: 758 (69.0%)
   With VALID coordinates: 757 (68.9%)

3. POTENTIAL RECORDS WITH BOTH COORDINATES:
   If matching was perfect, we could have ~336,335 records with coordinates
   (This is a very rough upper bound estimate)


In [29]:
# Check for ID format issues
print("\n4. SCHOOL ID FORMAT ANALYSIS:")

# Missing from coordinate file
origin_missing = origin_ids_raw - all_coord_school_ids
dest_missing = dest_ids_raw - all_coord_school_ids

print(f"\n   Origin schools NOT in coordinate file: {len(origin_missing):,}")
if len(origin_missing) > 0:
    print(f"   Examples:")
    for school_id in list(origin_missing)[:10]:
        matches = gr7_raw[gr7_raw['school_id_origin'] == school_id]
        if len(matches) > 0:
            school_name = matches['School Name in Grade 6'].iloc[0]
            print(f"     {school_id} - {school_name}")

print(f"\n   Destination schools NOT in coordinate file: {len(dest_missing):,}")
if len(dest_missing) > 0:
    print(f"   Examples:")
    for school_id in list(dest_missing)[:10]:
        matches = gr7_raw[gr7_raw['school_id_destination'] == school_id]
        if len(matches) > 0:
            school_name = matches['School Namein Grade 7'].iloc[0]
            print(f"     {school_id} - {school_name}")


4. SCHOOL ID FORMAT ANALYSIS:

   Origin schools NOT in coordinate file: 676
   Examples:
     411016 - Polytechnic College of La Union
     410126 - KINGSVILLE ADVANCED SCHOOL OF MALASIQUI INC.
     400553 - Ke Bing School
     101773 - Linmansangan ES
     402150 - La Verne Academy, Inc.
     400428 - Cordon Archangel's Montessori School, Inc.
     424340 - Amazing Learners and Molders Learning Center
     482799 - Saint Paul Educational Center
     416026 - PLT College, Inc.
     106687 - Timmaguab ES

   Destination schools NOT in coordinate file: 340
   Examples:
     411016 - Polytechnic College of La Union
     410126 - KINGSVILLE ADVANCED SCHOOL OF MALASIQUI INC.
     415063 - International School of Asia and the Pacific
     400032 - St. Agnes School, Inc.
     400259 - Escuela de Nuestra Señora de la Salette
     400146 - Binmaley Catholic School, Inc.
     400055 - Benito Soliven Academy, Inc.
     413004 - Bible Believing Baptist Church Educational Ministries Foundation, I

# 4. Process Grade 7 Enrollees with Module

**Goal:** Run through the actual processing pipeline and track coordinate matching

In [30]:
print("="*80)
print("STEP 4: PROCESS GRADE 7 ENROLLEES")
print("="*80)

# Initialize processor
gr7_processor = Gr7EnrolleesProcessor(
    public_nodes_path='output/public_nodes_valid.gpkg',
    private_nodes_path='output/private_nodes_valid.gpkg',
    enrollment_csv_path='data/processed/SY_2024_2025_School_Level_Data_on_Official_Enrollment.csv',
    verbose=True
)

# Process
gr7_data = gr7_processor.load_school_year('SY 2023-2024')
gr7_validated = gr7_processor.validate_against_nodes()

gr7_processed_data = gr7_processor.process(
    school_year='SY 2023-2024',
    export_path='output/gr7_enrollees_sy2023_2024.csv'
)

print(f"\n✓ Grade 7 enrollees processed")

INFO:modules.gr7_enrollees_processor:Gr7EnrolleesProcessor initialized
INFO:modules.gr7_enrollees_processor:Loading Grade 7 enrollees data for SY 2023-2024...
INFO:modules.gr7_enrollees_processor:  Reading SY 2023-2024 Gr 7 Enrollees.xlsx...


STEP 4: PROCESS GRADE 7 ENROLLEES


INFO:modules.gr7_enrollees_processor:  Loaded 1,034,848 student records
INFO:modules.gr7_enrollees_processor:  Standardizing school IDs...
INFO:modules.gr7_enrollees_processor:  School ID validation:
INFO:modules.gr7_enrollees_processor:    Origin IDs valid: 1,034,845/1,034,848 (100.0%)
INFO:modules.gr7_enrollees_processor:    Destination IDs valid: 1,034,848/1,034,848 (100.0%)
INFO:modules.gr7_enrollees_processor:    Both IDs valid: 1,034,845/1,034,848 (100.0%)
INFO:modules.gr7_enrollees_processor:Loading enrollment masterlist from SY_2024_2025_School_Level_Data_on_Official_Enrollment.csv...
INFO:modules.gr7_enrollees_processor:  Loaded 60,095 schools from enrollment masterlist
INFO:modules.gr7_enrollees_processor:Loading public nodes from public_nodes_valid.gpkg...
INFO:modules.gr7_enrollees_processor:  Loaded 44,899 public schools
INFO:modules.gr7_enrollees_processor:Loading private nodes from private_nodes_valid.gpkg...
INFO:modules.gr7_enrollees_processor:  Loaded 9,305 private sc


✓ Grade 7 enrollees processed


In [31]:
# Inspect the processed data
print("\nPROCESSED GRADE 7 ENROLLEES SUMMARY:")
print(f"  Total records: {len(gr7_validated):,}")
print(f"  Fully valid: {(gr7_validated['fully_valid'] == True).sum():,}")
print(f"  Both in node tables: {(gr7_validated['both_in_node_tables'] == True).sum():,}")

print(f"\n  Unique origin schools: {gr7_validated['school_id_origin'].nunique():,}")
print(f"  Unique destination schools: {gr7_validated['school_id_destination'].nunique():,}")

# Check school_id dtypes
print(f"\nSchool ID dtypes after processing:")
print(f"  school_id_origin: {gr7_validated['school_id_origin'].dtype}")
print(f"  school_id_destination: {gr7_validated['school_id_destination'].dtype}")

# Sample
print(f"\nSample processed records:")
display(gr7_validated[['lrn', 'school_id_origin', 'school_id_destination', 'sector_origin', 'sector_destination', 
                        'both_school_ids_valid', 'both_in_enrollment', 'both_in_node_tables', 'fully_valid']].head())


PROCESSED GRADE 7 ENROLLEES SUMMARY:
  Total records: 1,034,848
  Fully valid: 921,249
  Both in node tables: 924,109

  Unique origin schools: 30,631
  Unique destination schools: 9,381

School ID dtypes after processing:
  school_id_origin: object
  school_id_destination: object

Sample processed records:


,lrn,school_id_origin,school_id_destination,sector_origin,sector_destination,both_school_ids_valid,both_in_enrollment,both_in_node_tables,fully_valid
0,100160170039,100160,300001,public,public,True,True,True,True
1,100160160033,100160,300001,public,public,True,True,True,True
2,150007160008,100160,300001,public,public,True,True,True,True
3,100160160019,100160,300001,public,public,True,True,True,True
4,100160160020,100160,300001,public,public,True,True,True,True


# 5. DIAGNOSTIC: Node Table Inspection

**Goal:** Check if node tables have coordinates and understand the merge process

In [32]:
print("="*80)
print("STEP 5: NODE TABLE COORDINATE AVAILABILITY")
print("="*80)

# Load node tables
public_nodes = gpd.read_file('output/public_nodes_valid.gpkg')
private_nodes = gpd.read_file('output/private_nodes_valid.gpkg')

# Standardize school_id
public_nodes['school_id'] = public_nodes['school_id'].astype(str).str.strip()
private_nodes['school_id'] = private_nodes['school_id'].astype(str).str.strip()

STEP 5: NODE TABLE COORDINATE AVAILABILITY


In [33]:
public_nodes.columns

Index(['school_id', 'latitude', 'longitude', 'coordinates_valid',
       'enrollment_es', 'enrollment_jhs', 'enrollment_shs',
       'has_enrollment_data', 'offers_es', 'offers_jhs', 'offers_shs',
       'es_classrooms_instructional', 'es_classrooms_non_instructional',
       'jhs_classrooms_instructional', 'jhs_classrooms_non_instructional',
       'shs_classrooms_instructional', 'shs_classrooms_non_instructional',
       'has_facilities_data', 'seats_es', 'seats_jhs', 'seats_shs',
       'has_seats_data', 'adm2_pcode', 'adm1_pcode', 'region', 'province',
       'adm3_psgc', 'municipality', 'admin_assignment_valid',
       'total_enrollment', 'total_seats', 'capacity_utilization',
       'validation_level_1', 'validation_level_2', 'validation_level_3',
       'all_valid', 'geometry'],
      dtype='object')

In [34]:
print(f"\n1. PUBLIC NODE TABLE:")
print(f"   Total schools: {len(public_nodes):,}")
print(f"   With valid coordinates: {(public_nodes['coordinates_valid'] == True).sum():,} ({(public_nodes['coordinates_valid'] == True).sum()/len(public_nodes)*100:.1f}%)")
print(f"   With latitude: {public_nodes['latitude'].notna().sum():,}")
print(f"   With longitude: {public_nodes['longitude'].notna().sum():,}")

print(f"\n2. PRIVATE NODE TABLE:")
print(f"   Total schools: {len(private_nodes):,}")
print(f"   With valid coordinates: {(private_nodes['coordinates_valid'] == True).sum():,} ({(private_nodes['coordinates_valid'] == True).sum()/len(private_nodes)*100:.1f}%)")
print(f"   With latitude: {private_nodes['latitude'].notna().sum():,}")
print(f"   With longitude: {private_nodes['longitude'].notna().sum():,}")

# Sample schools with coordinates
print(f"\n3. SAMPLE PUBLIC SCHOOLS WITH COORDINATES:")
sample_pub = public_nodes[public_nodes['coordinates_valid'] == True][['school_id', 'latitude', 'longitude']].head(3)
display(sample_pub)

print(f"\n4. SAMPLE PUBLIC SCHOOLS WITHOUT COORDINATES:")
sample_pub_no = public_nodes[public_nodes['coordinates_valid'] != True][['school_id', 'latitude', 'longitude']].head(3)
display(sample_pub_no)


1. PUBLIC NODE TABLE:
   Total schools: 44,899
   With valid coordinates: 44,899 (100.0%)
   With latitude: 44,899
   With longitude: 44,899

2. PRIVATE NODE TABLE:
   Total schools: 9,305
   With valid coordinates: 9,305 (100.0%)
   With latitude: 9,305
   With longitude: 9,305

3. SAMPLE PUBLIC SCHOOLS WITH COORDINATES:


,school_id,latitude,longitude
0,100001,18.266860,120.614372
1,100002,18.251272,120.609487
2,100003,18.234670,120.616050



4. SAMPLE PUBLIC SCHOOLS WITHOUT COORDINATES:


,school_id,latitude,longitude


In [35]:
# Check overlap between node tables and Grade 7 enrollees
print("\n5. NODE TABLE vs GRADE 7 ENROLLEES OVERLAP:")

public_node_ids = set(public_nodes['school_id'].unique())
private_node_ids = set(private_nodes['school_id'].unique())
all_node_ids = public_node_ids | private_node_ids

public_with_coords = set(public_nodes[public_nodes['coordinates_valid'] == True]['school_id'].unique())
private_with_coords = set(private_nodes[private_nodes['coordinates_valid'] == True]['school_id'].unique())
all_with_coords = public_with_coords | private_with_coords

# Compare with Grade 7 enrollees
gr7_origin_ids = set(gr7_validated['school_id_origin'].dropna().unique())
gr7_dest_ids = set(gr7_validated['school_id_destination'].dropna().unique())

print(f"\n   Origin schools in node tables: {len(gr7_origin_ids & all_node_ids):,}/{len(gr7_origin_ids):,} ({len(gr7_origin_ids & all_node_ids)/len(gr7_origin_ids)*100:.1f}%)")
print(f"   Origin schools with coordinates: {len(gr7_origin_ids & all_with_coords):,}/{len(gr7_origin_ids):,} ({len(gr7_origin_ids & all_with_coords)/len(gr7_origin_ids)*100:.1f}%)")

print(f"\n   Destination schools in node tables: {len(gr7_dest_ids & all_node_ids):,}/{len(gr7_dest_ids):,} ({len(gr7_dest_ids & all_node_ids)/len(gr7_dest_ids)*100:.1f}%)")
print(f"   Destination schools with coordinates: {len(gr7_dest_ids & all_with_coords):,}/{len(gr7_dest_ids):,} ({len(gr7_dest_ids & all_with_coords)/len(gr7_dest_ids)*100:.1f}%)")


5. NODE TABLE vs GRADE 7 ENROLLEES OVERLAP:

   Origin schools in node tables: 27,582/30,631 (90.0%)
   Origin schools with coordinates: 27,582/30,631 (90.0%)

   Destination schools in node tables: 8,349/9,381 (89.0%)
   Destination schools with coordinates: 8,349/9,381 (89.0%)


# 6. Build Unified Flows with Dtype Fix Applied

**Goal:** Build unified flows with the dtype fix to verify improved coordinate coverage

**IMPORTANT:** This section reloads the `unified_gr7_flow_builder` module to ensure the dtype fix is active.

In [19]:
print("="*80)
print("STEP 6: BUILD UNIFIED FLOWS (WITH DTYPE FIX)")
print("="*80)

# CRITICAL: Reload module to get dtype fix
print("\n🔧 Reloading unified_gr7_flow_builder module to apply dtype fix...")
import importlib
importlib.reload(unified_gr7_flow_builder)
from modules.unified_gr7_flow_builder import UnifiedGr7FlowBuilder

# Initialize flow builder
flow_builder = UnifiedGr7FlowBuilder(
    beneficiary_parquet_path='data/processed/esc_beneficiaries.parquet',
    gr7_enrollees_path='output/gr7_enrollees_sy2023_2024.csv',
    public_nodes_path='output/public_nodes_valid.gpkg',
    private_nodes_path='output/private_nodes_valid.gpkg',
    enrollment_csv_path='data/processed/SY_2024_2025_School_Level_Data_on_Official_Enrollment.csv',
    verbose=True
)

# Build unified flows with fix
unified_flows = flow_builder.build_unified_table(school_year='SY 2023-2024')

print(f"\n✓ Unified flows built with dtype fix")

INFO:modules.unified_gr7_flow_builder:UnifiedGr7FlowBuilder initialized
INFO:modules.unified_gr7_flow_builder:============================================================
INFO:modules.unified_gr7_flow_builder:Building unified Grade 7 flow table for SY 2023-2024
INFO:modules.unified_gr7_flow_builder:============================================================
INFO:modules.unified_gr7_flow_builder:Step 1/7: Loading beneficiary data...


STEP 6: BUILD UNIFIED FLOWS (WITH DTYPE FIX)

🔧 Reloading unified_gr7_flow_builder module to apply dtype fix...


INFO:modules.unified_gr7_flow_builder:  Loaded 2,681,668 total beneficiary records
INFO:modules.unified_gr7_flow_builder:  Grade distribution: {np.int64(7): 713178, np.int64(8): 650467, np.int64(9): 648316, np.int64(10): 669707}
INFO:modules.unified_gr7_flow_builder:  Filtered to 226,075 Grade 7 beneficiaries in SY 2023-2024
INFO:modules.unified_gr7_flow_builder:  Unique beneficiary students (LRNs): 226,075
INFO:modules.unified_gr7_flow_builder:Step 2/7: Loading Gr7 enrollees data...
INFO:modules.unified_gr7_flow_builder:  Loaded 1,034,848 Gr7 enrollee records
INFO:modules.unified_gr7_flow_builder:  Filtered to 921,249 fully valid records
INFO:modules.unified_gr7_flow_builder:  Unique Gr7 students (LRNs): 921,249
INFO:modules.unified_gr7_flow_builder:Step 3/8: Merging datasets by LRN...
INFO:modules.unified_gr7_flow_builder:  Beneficiary LRNs: 226,075
INFO:modules.unified_gr7_flow_builder:  Gr7 enrollees LRNs: 921,249
INFO:modules.unified_gr7_flow_builder:  Merge results:
INFO:modules.


✓ Unified flows built with dtype fix


In [20]:
print("\n" + "="*80)
print("COORDINATE COVERAGE VERIFICATION (AFTER DTYPE FIX)")
print("="*80)

print("\nUNIFIED FLOWS SUMMARY:")
print(f"  Total records: {len(unified_flows):,}")
print(f"  Records with origin coordinates: {unified_flows['latitude_origin'].notna().sum():,} ({unified_flows['latitude_origin'].notna().sum()/len(unified_flows)*100:.1f}%)")
print(f"  Records with destination coordinates: {unified_flows['latitude_destination'].notna().sum():,} ({unified_flows['latitude_destination'].notna().sum()/len(unified_flows)*100:.1f}%)")

both_coords_count = (unified_flows['latitude_origin'].notna() & unified_flows['latitude_destination'].notna()).sum()
both_coords_pct = both_coords_count / len(unified_flows) * 100

print(f"  Records with BOTH coordinates: {both_coords_count:,} ({both_coords_pct:.1f}%)")

# Check coordinate column dtypes
print(f"\nCoordinate column dtypes:")
print(f"  latitude_origin: {unified_flows['latitude_origin'].dtype}")
print(f"  longitude_origin: {unified_flows['longitude_origin'].dtype}")
print(f"  latitude_destination: {unified_flows['latitude_destination'].dtype}")
print(f"  longitude_destination: {unified_flows['longitude_destination'].dtype}")

# Check school_id dtypes
print(f"\nSchool ID dtypes:")
print(f"  school_id_origin: {unified_flows['school_id_origin'].dtype}")
print(f"  school_id_destination: {unified_flows['school_id_destination'].dtype}")

# Sample school IDs to check for float artifacts
origin_sample = unified_flows['school_id_origin'].dropna().head(5).tolist()
dest_sample = unified_flows['school_id_destination'].dropna().head(5).tolist()
print(f"  Sample origin IDs: {origin_sample}")
print(f"  Sample dest IDs: {dest_sample}")

# Success check
print(f"\n{'='*80}")
if both_coords_pct > 75:
    print(f"✅ SUCCESS! Coordinate coverage is {both_coords_pct:.1f}% (expected ~80-85%)")
    print(f"   Improvement from 5.5% → {both_coords_pct:.1f}% ({both_coords_pct - 5.5:.1f} percentage points)")
elif both_coords_pct > 50:
    print(f"⚠️  PARTIAL FIX: Coordinate coverage is {both_coords_pct:.1f}%")
    print(f"   Expected ~80-85%, but this is better than 5.5%")
    print(f"   Further investigation may be needed.")
else:
    print(f"❌ FIX NOT WORKING: Coordinate coverage still only {both_coords_pct:.1f}%")
    print(f"   Expected ~80-85%. Dtype fix may not be applied correctly.")
print(f"{'='*80}")


COORDINATE COVERAGE VERIFICATION (AFTER DTYPE FIX)

UNIFIED FLOWS SUMMARY:
  Total records: 1,043,248
  Records with origin coordinates: 1,010,834 (96.9%)
  Records with destination coordinates: 992,615 (95.1%)
  Records with BOTH coordinates: 979,206 (93.9%)

Coordinate column dtypes:
  latitude_origin: float64
  longitude_origin: float64
  latitude_destination: float64
  longitude_destination: float64

School ID dtypes:
  school_id_origin: object
  school_id_destination: object
  Sample origin IDs: ['100001', '100001', '100001', '100001', '100001']
  Sample dest IDs: ['300002', '300002', '300002', '300002', '300002']

✅ SUCCESS! Coordinate coverage is 93.9% (expected ~80-85%)
   Improvement from 5.5% → 93.9% (88.4 percentage points)


In [21]:
with pd.option_context('display.max_columns', None, 'display.max_rows', None):
    display(unified_flows.head(1))

,region,division,lrn,school_id_destination,school_name_destination,sy_grade6,school_id_origin,school_name_origin,school_year,school_id_origin_valid,school_id_destination_valid,both_school_ids_valid,sector_origin,municipality_origin,sector_destination,municipality_destination,origin_in_public_nodes,origin_in_private_nodes,origin_in_node_tables,destination_in_public_nodes,destination_in_private_nodes,destination_in_node_tables,both_in_node_tables,both_in_enrollment,fully_valid,esc_subsidy_amount,is_beneficiary,latitude_origin,longitude_origin,latitude_destination,longitude_destination,distance_straightline_km,flow_type
0,Region I,Ilocos Norte,100001160001,300002,Bacarra NCHS,2022.0,100001,Apaleng-Libtong PS,SY 2023-2024,True,True,True,public,BACARRA,public,BACARRA,True,False,True,True,False,True,True,True,True,NaN,False,18.26686,120.614372,18.250214,120.613077,1.855995,public_to_public_nonbeneficiary


## 6.1 Export fixed flows

In [24]:
# Filter by fully_valid status
valid_flows = unified_flows[unified_flows['fully_valid'] == True]
all_flows = unified_flows

In [23]:
print(valid_flows.shape)
with pd.option_context('display.max_columns', None, 'display.max_rows', None):
    display(valid_flows.head(3))

display(valid_flows.dtypes)

(922059, 33)


,region,division,lrn,school_id_destination,school_name_destination,sy_grade6,school_id_origin,school_name_origin,school_year,school_id_origin_valid,school_id_destination_valid,both_school_ids_valid,sector_origin,municipality_origin,sector_destination,municipality_destination,origin_in_public_nodes,origin_in_private_nodes,origin_in_node_tables,destination_in_public_nodes,destination_in_private_nodes,destination_in_node_tables,both_in_node_tables,both_in_enrollment,fully_valid,esc_subsidy_amount,is_beneficiary,latitude_origin,longitude_origin,latitude_destination,longitude_destination,distance_straightline_km,flow_type
0,Region I,Ilocos Norte,100001160001,300002,Bacarra NCHS,2022.0,100001,Apaleng-Libtong PS,SY 2023-2024,True,True,True,public,BACARRA,public,BACARRA,True,False,True,True,False,True,True,True,True,NaN,False,18.26686,120.614372,18.250214,120.613077,1.855995,public_to_public_nonbeneficiary
1,Region I,Ilocos Norte,100001160002,300002,Bacarra NCHS,2022.0,100001,Apaleng-Libtong PS,SY 2023-2024,True,True,True,public,BACARRA,public,BACARRA,True,False,True,True,False,True,True,True,True,NaN,False,18.26686,120.614372,18.250214,120.613077,1.855995,public_to_public_nonbeneficiary
2,Region I,Ilocos Norte,100001160003,300002,Bacarra NCHS,2022.0,100001,Apaleng-Libtong PS,SY 2023-2024,True,True,True,public,BACARRA,public,BACARRA,True,False,True,True,False,True,True,True,True,NaN,False,18.26686,120.614372,18.250214,120.613077,1.855995,public_to_public_nonbeneficiary


region                           object
division                         object
lrn                              object
school_id_destination            object
school_name_destination          object
sy_grade6                       float64
school_id_origin                 object
school_name_origin               object
school_year                      object
school_id_origin_valid           object
school_id_destination_valid      object
both_school_ids_valid            object
sector_origin                    object
municipality_origin              object
sector_destination               object
municipality_destination         object
origin_in_public_nodes           object
origin_in_private_nodes          object
origin_in_node_tables            object
destination_in_public_nodes      object
destination_in_private_nodes     object
destination_in_node_tables       object
both_in_node_tables              object
both_in_enrollment               object
fully_valid                      object


In [25]:
# Export to separate CSV files
valid_flows.to_csv('output/gr7_enrollees_sy2023_2024_valid.csv', index=False)
all_flows.to_csv('output/gr7_enrollees_sy2023_2024_all.csv', index=False)

# 7. DEEP DIVE: Why Are Coordinates Missing?

**Goal:** Systematically identify the bottleneck

In [29]:
print("="*80)
print("STEP 7: ROOT CAUSE ANALYSIS")
print("="*80)

# Create analysis subsets
has_both_coords = unified_flows[
    unified_flows['latitude_origin'].notna() & 
    unified_flows['latitude_destination'].notna()
]

missing_origin = unified_flows[
    unified_flows['latitude_origin'].isna() &
    unified_flows['latitude_destination'].notna()
]

missing_dest = unified_flows[
    unified_flows['latitude_origin'].notna() &
    unified_flows['latitude_destination'].isna()
]

missing_both = unified_flows[
    unified_flows['latitude_origin'].isna() &
    unified_flows['latitude_destination'].isna()
]

print(f"\n1. COORDINATE AVAILABILITY BREAKDOWN:")
print(f"   Both coordinates: {len(has_both_coords):,} ({len(has_both_coords)/len(unified_flows)*100:.1f}%)")
print(f"   Only origin: {len(missing_dest):,} ({len(missing_dest)/len(unified_flows)*100:.1f}%)")
print(f"   Only destination: {len(missing_origin):,} ({len(missing_origin)/len(unified_flows)*100:.1f}%)")
print(f"   Neither: {len(missing_both):,} ({len(missing_both)/len(unified_flows)*100:.1f}%)")

STEP 7: ROOT CAUSE ANALYSIS

1. COORDINATE AVAILABILITY BREAKDOWN:
   Both coordinates: 57,147 (5.5%)
   Only origin: 31,628 (3.0%)
   Only destination: 13,409 (1.3%)
   Neither: 940,254 (90.2%)


In [30]:
# Check if missing coordinates correlate with validation flags
print("\n2. CORRELATION WITH VALIDATION FLAGS:")

print(f"\n   Records with BOTH coordinates:")
if len(has_both_coords) > 0:
    print(f"     both_in_node_tables=True: {(has_both_coords['both_in_node_tables'] == True).sum():,} ({(has_both_coords['both_in_node_tables'] == True).sum()/len(has_both_coords)*100:.1f}%)")
    print(f"     fully_valid=True: {(has_both_coords['fully_valid'] == True).sum():,} ({(has_both_coords['fully_valid'] == True).sum()/len(has_both_coords)*100:.1f}%)")

print(f"\n   Records missing BOTH coordinates:")
if len(missing_both) > 0:
    print(f"     both_in_node_tables=True: {(missing_both['both_in_node_tables'] == True).sum():,} ({(missing_both['both_in_node_tables'] == True).sum()/len(missing_both)*100:.1f}%)")
    print(f"     fully_valid=True: {(missing_both['fully_valid'] == True).sum():,}")


2. CORRELATION WITH VALIDATION FLAGS:

   Records with BOTH coordinates:
     both_in_node_tables=True: 0 (0.0%)
     fully_valid=True: 0 (0.0%)

   Records missing BOTH coordinates:
     both_in_node_tables=True: 921,249 (98.0%)
     fully_valid=True: 921,249


In [31]:
# Check specific examples
print("\n3. EXAMPLE: Records in node tables but WITHOUT coordinates:")

# Filter to records that should have coordinates but don't
in_nodes_no_coords = unified_flows[
    (unified_flows['both_in_node_tables'] == True) &
    (unified_flows['latitude_origin'].isna() | unified_flows['latitude_destination'].isna())
]

print(f"   Count: {len(in_nodes_no_coords):,}")

if len(in_nodes_no_coords) > 0:
    print(f"\n   First 5 examples:")
    cols_to_show = [
        'lrn', 'school_id_origin', 'school_id_destination',
        'school_name_origin', 'school_name_destination',
        'both_in_node_tables', 'latitude_origin', 'latitude_destination'
    ]
    display(in_nodes_no_coords[cols_to_show].head())


3. EXAMPLE: Records in node tables but WITHOUT coordinates:
   Count: 921,249

   First 5 examples:


,lrn,school_id_origin,school_id_destination,school_name_origin,school_name_destination,both_in_node_tables,latitude_origin,latitude_destination
0,100001160001,100001.0,300002.0,Apaleng-Libtong PS,Bacarra NCHS,True,NaN,NaN
1,100001160002,100001.0,300002.0,Apaleng-Libtong PS,Bacarra NCHS,True,NaN,NaN
2,100001160003,100001.0,300002.0,Apaleng-Libtong PS,Bacarra NCHS,True,NaN,NaN
3,100001160004,100001.0,300002.0,Apaleng-Libtong PS,Bacarra NCHS,True,NaN,NaN
4,100001160005,100001.0,300002.0,Apaleng-Libtong PS,Bacarra NCHS,True,NaN,NaN


In [32]:
# Manual verification: Check if these schools are actually in node tables
if len(in_nodes_no_coords) > 0:
    print("\n4. MANUAL VERIFICATION: Are these schools really in node tables?")
    
    # Take first example
    example = in_nodes_no_coords.iloc[0]
    origin_id = str(example['school_id_origin']).strip()
    dest_id = str(example['school_id_destination']).strip()
    
    print(f"\n   Example origin school ID: {origin_id}")
    print(f"   Example destination school ID: {dest_id}")
    
    # Check in node tables
    origin_in_public = public_nodes[public_nodes['school_id'] == origin_id]
    origin_in_private = private_nodes[private_nodes['school_id'] == origin_id]
    
    dest_in_public = public_nodes[public_nodes['school_id'] == dest_id]
    dest_in_private = private_nodes[private_nodes['school_id'] == dest_id]
    
    print(f"\n   Origin school in public nodes: {len(origin_in_public) > 0}")
    if len(origin_in_public) > 0:
        print(f"     - Has coordinates: {origin_in_public.iloc[0]['coordinates_valid']}")
        print(f"     - Latitude: {origin_in_public.iloc[0]['latitude']}")
        print(f"     - Longitude: {origin_in_public.iloc[0]['longitude']}")
    
    print(f"\n   Origin school in private nodes: {len(origin_in_private) > 0}")
    if len(origin_in_private) > 0:
        print(f"     - Has coordinates: {origin_in_private.iloc[0]['coordinates_valid']}")
        print(f"     - Latitude: {origin_in_private.iloc[0]['latitude']}")
    
    print(f"\n   Destination school in public nodes: {len(dest_in_public) > 0}")
    if len(dest_in_public) > 0:
        print(f"     - Has coordinates: {dest_in_public.iloc[0]['coordinates_valid']}")
        print(f"     - Latitude: {dest_in_public.iloc[0]['latitude']}")
    
    print(f"\n   Destination school in private nodes: {len(dest_in_private) > 0}")
    if len(dest_in_private) > 0:
        print(f"     - Has coordinates: {dest_in_private.iloc[0]['coordinates_valid']}")


4. MANUAL VERIFICATION: Are these schools really in node tables?

   Example origin school ID: 100001.0
   Example destination school ID: 300002.0

   Origin school in public nodes: False

   Origin school in private nodes: False

   Destination school in public nodes: False

   Destination school in private nodes: False


## 7.1 Important note
It seems that the data types of school IDs are transformed to float that may account for the ~5.1% match rate between the node tables and the grade 7 enrollment.

In [35]:
# Display result of example school ID when in integer - as seen in the validation code
display(public_nodes[public_nodes['school_id'] == 100001])

,school_id,latitude,longitude,coordinates_valid,enrollment_es,enrollment_jhs,enrollment_shs,has_enrollment_data,offers_es,offers_jhs,...,municipality,admin_assignment_valid,total_enrollment,total_seats,capacity_utilization,validation_level_1,validation_level_2,validation_level_3,all_valid,geometry


In [36]:
# Display result of example school ID when in float - as seen in the validation code
display(public_nodes[public_nodes['school_id'] == 100001.0])

,school_id,latitude,longitude,coordinates_valid,enrollment_es,enrollment_jhs,enrollment_shs,has_enrollment_data,offers_es,offers_jhs,...,municipality,admin_assignment_valid,total_enrollment,total_seats,capacity_utilization,validation_level_1,validation_level_2,validation_level_3,all_valid,geometry


In [34]:
# Display result of example school ID when in string
display(public_nodes[public_nodes['school_id'] == '100001'])

,school_id,latitude,longitude,coordinates_valid,enrollment_es,enrollment_jhs,enrollment_shs,has_enrollment_data,offers_es,offers_jhs,...,municipality,admin_assignment_valid,total_enrollment,total_seats,capacity_utilization,validation_level_1,validation_level_2,validation_level_3,all_valid,geometry
0,100001,18.26686,120.614372,True,49.0,0.0,0.0,True,True,False,...,Bacarra,True,49.0,195.0,0.251282,True,True,True,True,POINT (120.61437 18.26686)


# 8. Post-Fix Analysis: Inspect Flow Builder Logic

**Goal:** Inspect the flow builder's internal state to verify the dtype fix is working correctly

In [ ]:
print("="*80)
print("TESTING DTYPE FIX")
print("="*80)

# Reload the module with fixes
import importlib
importlib.reload(unified_gr7_flow_builder)
from modules.unified_gr7_flow_builder import UnifiedGr7FlowBuilder

# Build unified flows WITH the fix
print("\n🔧 Building unified flows with dtype fix applied...")
flow_builder_fixed = UnifiedGr7FlowBuilder(
    beneficiary_parquet_path='data/processed/esc_beneficiaries.parquet',
    gr7_enrollees_path='output/gr7_enrollees_sy2023_2024.csv',
    public_nodes_path='output/public_nodes_valid.gpkg',
    private_nodes_path='output/private_nodes_valid.gpkg',
    enrollment_csv_path='data/processed/SY_2024_2025_School_Level_Data_on_Official_Enrollment.csv',
    verbose=True
)

unified_flows_fixed = flow_builder_fixed.build_unified_table(school_year='SY 2023-2024')

print("\n✅ Unified flows rebuilt with dtype fix")

In [ ]:
print("="*80)
print("STEP 8: INSPECT FLOW BUILDER MERGE LOGIC")
print("="*80)

# Access the node tables loaded by flow builder
print(f"\n1. NODE TABLES LOADED BY FLOW BUILDER:")
print(f"   Public nodes: {len(flow_builder.public_nodes):,} schools")
print(f"   Private nodes: {len(flow_builder.private_nodes):,} schools")

# Check what columns they have
print(f"\n2. PUBLIC NODE TABLE COLUMNS:")
print(f"   {list(flow_builder.public_nodes.columns)}")

print(f"\n3. PRIVATE NODE TABLE COLUMNS:")
print(f"   {list(flow_builder.private_nodes.columns)}")

# Check for coordinate columns
print(f"\n4. COORDINATE COLUMNS IN NODE TABLES:")
has_lat = 'latitude' in flow_builder.public_nodes.columns
has_lon = 'longitude' in flow_builder.public_nodes.columns
print(f"   Public nodes has 'latitude': {has_lat}")
print(f"   Public nodes has 'longitude': {has_lon}")

if has_lat:
    pub_with_lat = flow_builder.public_nodes['latitude'].notna().sum()
    print(f"   Public nodes with non-null latitude: {pub_with_lat:,} ({pub_with_lat/len(flow_builder.public_nodes)*100:.1f}%)")

## 7.2 Dtype Fix Summary

**Fix Applied in Section 6:** School ID dtype conversion in UnifiedGr7FlowBuilder

**Three-Part Fix:**
1. **Fix 1:** Conditional dtype conversion in `_merge_by_lrn()` after outer merge
   - If numeric (float64/int64): Convert `float64 → Int64 → string`
   - If already string (object): Clean up 'None', 'nan', '<NA>' artifacts
2. **Fix 2:** Validation and conversion in `_add_school_attributes()` before coordinate merge
   - Safety net to catch any numeric dtypes that slipped through
3. **Fix 3:** Logging to track dtype changes and verify no float artifacts
   - Logs before/after dtypes, sample IDs, and node table dtypes

**Expected Outcome:** Coordinate coverage increases from 5.5% to ~80-85%

**See Section 6 results above for actual improvement.**

In [ ]:
print("="*80)
print("DIAGNOSTIC SUMMARY AND RECOMMENDATIONS")
print("="*80)

print(f"\n📊 KEY FINDINGS:")
print(f"\n1. Coordinate Source Coverage:")
print(f"   - Coordinate file has ~{len(all_coord_school_ids):,} schools")
print(f"   - Valid coordinates: ~{len(valid_coord_school_ids):,} schools")

print(f"\n2. Grade 7 Enrollees:")
print(f"   - Origin schools: ~{len(gr7_origin_ids):,}")
print(f"   - Destination schools: ~{len(gr7_dest_ids):,}")
print(f"   - Origin schools with valid coords: ~{len(gr7_origin_ids & all_with_coords):,} ({len(gr7_origin_ids & all_with_coords)/len(gr7_origin_ids)*100:.1f}%)")
print(f"   - Dest schools with valid coords: ~{len(gr7_dest_ids & all_with_coords):,} ({len(gr7_dest_ids & all_with_coords)/len(gr7_dest_ids)*100:.1f}%)")

print(f"\n3. Unified Flows Result:")
print(f"   - Total records: {len(unified_flows):,}")
print(f"   - Records with both coordinates: {len(has_both_coords):,} ({len(has_both_coords)/len(unified_flows)*100:.1f}%)")

print(f"\n4. In node tables but no coordinates: {len(in_nodes_no_coords):,} records")

print(f"\n💡 LIKELY ROOT CAUSES:")
print(f"   [ ] Grade 6 elementary schools poorly covered in coordinate source")
print(f"   [ ] School ID format mismatch between datasets")
print(f"   [ ] Node table merge not bringing coordinates properly")
print(f"   [ ] Coordinate columns exist in node tables but have NaN values")
print(f"   [ ] Filter somewhere removing schools with coordinates")

print(f"\n🔧 NEXT STEPS:")
print(f"   1. Review above diagnostics to identify which cause is active")
print(f"   2. Check UnifiedGr7FlowBuilder._add_school_attributes() merge logic")
print(f"   3. Verify node tables were built with coordinates included")
print(f"   4. Consider alternative coordinate sources for elementary schools")

# 9. Summary and Next Steps

**Findings based on diagnostics above:**

In [ ]:
# Export unified flows
flow_builder.export('output/unified_gr7_flows_sy2023_2024.csv')
print("✓ Unified flows exported to: output/unified_gr7_flows_sy2023_2024.csv")

# Export diagnostic summary
diagnostic_summary = {
    'total_records': len(unified_flows),
    'with_both_coordinates': len(has_both_coords),
    'with_both_coords_pct': len(has_both_coords)/len(unified_flows)*100,
    'origin_schools_total': len(gr7_origin_ids),
    'origin_schools_with_coords': len(gr7_origin_ids & all_with_coords),
    'dest_schools_total': len(gr7_dest_ids),
    'dest_schools_with_coords': len(gr7_dest_ids & all_with_coords),
    'coord_file_schools': len(all_coord_school_ids),
    'coord_file_valid': len(valid_coord_school_ids)
}

with open('output/coordinate_diagnostic_summary.json', 'w') as f:
    json.dump(diagnostic_summary, f, indent=2)

print("✓ Diagnostic summary exported to: output/coordinate_diagnostic_summary.json")